In [52]:
import scanpy as sc
import pandas as pd
import hotspot
import numpy as np
import joblib
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import leaves_list
import plotly.express as px

In [3]:
adata = sc.read("/dfs3b/ruic20_lab/tingty7/projects/ocular_surface/scRNA_wo_fetal/RNA_velocity/epithelial_velo.h5ad")
adata

AnnData object with n_obs × n_vars = 171184 × 36601
    obs: 'Unnamed: 0', 'barcode', 'leiden_0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'pANN', 'DF.classifications', 'tissue', 'donor', 'age', 'gender', 'race', 'study', 'sampleid', 'published_annotation', 'majorclass', 'celltype', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'n_counts', 'velocity_self_transition', 'root_cells', 'end_points', 'velocity_pseudotime', 'latent_time'
    obsm: 'X_scVI', 'X_umap'

In [5]:
adata.X.data

array([ 1.,  1.,  1., ...,  6., 32.,  2.], dtype=float32)

In [6]:
adata.layers["counts"] = adata.X

In [10]:
adata.obsm["velocity_pseudotime"] = np.asarray([[x] for x in adata.obs["velocity_pseudotime"]])

In [13]:
sc.pp.calculate_qc_metrics(adata, inplace=True)
adata = adata[:, adata.var.mean_counts > 0]


In [17]:
sc.pp.highly_variable_genes(
        adata, flavor="seurat_v3", n_top_genes=10000, subset=True
    )

/opt/apps/python/3.10.2/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:163: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["hvg"] = {"flavor": flavor}


In [19]:
sc.pp.subsample(adata, n_obs=10000, random_state=0)

In [21]:
adata

AnnData object with n_obs × n_vars = 10000 × 10000
    obs: 'Unnamed: 0', 'barcode', 'leiden_0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'pANN', 'DF.classifications', 'tissue', 'donor', 'age', 'gender', 'race', 'study', 'sampleid', 'published_annotation', 'majorclass', 'celltype', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'n_counts', 'velocity_self_transition', 'root_cells', 'end_points', 'velocity_pseudotime', 'latent_time', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg'
    obsm: 'X_scVI', 'X_umap', 'velocity_pseudotime'
    layers: 'counts'

In [23]:
sc.pp.calculate_qc_metrics(adata, inplace=True)
adata = adata[:, adata.var.mean_counts > 0]
adata

View of AnnData object with n_obs × n_vars = 10000 × 8763
    obs: 'Unnamed: 0', 'barcode', 'leiden_0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'pANN', 'DF.classifications', 'tissue', 'donor', 'age', 'gender', 'race', 'study', 'sampleid', 'published_annotation', 'majorclass', 'celltype', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'n_counts', 'velocity_self_transition', 'root_cells', 'end_points', 'velocity_pseudotime', 'latent_time', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg'
    obsm: 'X_scVI', 'X_umap', 'velocity_pseudotime'
    layers: 'counts'

In [24]:
hs = hotspot.Hotspot(
    adata,
    model="danb",
    layer_key="counts",
    latent_obsm_key="velocity_pseudotime",
    umi_counts_obs_key="total_counts",
)

/home/jovyan/.local/lib/python3.10/site-packages/hotspot/hotspot.py:98: UserWarning: Hotspot will work faster when counts are a csr sparse matrix.
  warnings.warn(


In [25]:
hs.create_knn_graph(weighted_graph=False, n_neighbors=30)
hs_results = hs.compute_autocorrelations(jobs=12)

hs_genes = hs_results.loc[hs_results.FDR < 0.05].index  # Select genes
local_correlations = hs.compute_local_correlations(
    hs_genes, jobs=12
)  # jobs for parallelization

modules = hs.create_modules(min_gene_threshold=250, core_only=True, fdr_threshold=0.05)
module_scores = hs.calculate_module_scores()


100%|██████████| 8763/8763 [01:51<00:00, 78.55it/s]


Computing pair-wise local correlation on 4389 features...


100%|██████████| 9629466/9629466 [36:35<00:00, 4386.49it/s] 


Computing scores for 5 modules...


100%|██████████| 5/5 [00:09<00:00,  1.82s/it]


In [26]:
output_path = "/dfs3b/ruic20_lab/tingty7/projects/ocular_surface/scRNA_wo_fetal/RNA_velocity/hotspot/"

In [29]:

local_correlations.to_csv(
    output_path + "epi_hs_local_correlations_velocity_pseudotime.csv"
)
module_scores.to_csv(
    output_path + "epi_hs_module_scores_velocity_pseudotime.csv"
)
joblib.dump(
    hs,
    output_path + "epi_hs.create_modules_velocity_pseudotime.pkl",
)
adata.write(output_path + "epi_hs.adata_velocity_pseudotime.h5ad"
)